# Análisis de los errores en el test

El test ya se evaluó una sola vez con `src/evaluate.py`. Acá no se entrena ni se cambia nada:
solo buscamos por qué la tasa de falsos positivos de XGBoost sube de 0,080 en la cross-validation a 0,251 en el test.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # para importar src desde notebooks/

import joblib
import pandas as pd

from src.config import MODELS_DIR
from src.data import get_X_y, load_split

train = load_split("train")
test = load_split("test")

## 1. Predicciones del modelo binario congelado sobre el test

In [2]:
bundle = joblib.load(MODELS_DIR / "binary_xgb.joblib")
X_test, y_test = get_X_y(test, "binary")
score = bundle["pipeline"].predict_proba(X_test)[:, 1]
pred = (score >= bundle["threshold"]).astype(int)

normales_test = test[y_test == 0].copy()
normales_test["falso_positivo"] = pred[y_test.values == 0]
print("Normales en el test:", len(normales_test))
print("Falsos positivos:", normales_test["falso_positivo"].sum())
print("Tasa de falsos positivos:", round(normales_test["falso_positivo"].mean(), 3))

Normales en el test: 37000
Falsos positivos: 9283
Tasa de falsos positivos: 0.251

## 2. ¿Qué variable concentra los falsos positivos?

Comparamos cada valor de `sttl` (TTL de los paquetes del origen): qué proporción del tráfico normal tiene en el train
y en el test, y qué tasa de falsos positivos genera en el test.

In [3]:
normales_train = train[train["label"] == 0]
tabla = normales_test.groupby("sttl").agg(n_test=("falso_positivo", "size"), falsos_positivos=("falso_positivo", "sum"))
tabla["prop_test"] = tabla["n_test"] / len(normales_test)
tabla["tasa_fp"] = tabla["falsos_positivos"] / tabla["n_test"]
tabla["prop_train"] = normales_train["sttl"].value_counts(normalize=True)
tabla.fillna(0).sort_values("falsos_positivos", ascending=False).head(6).round(3)

,n_test,falsos_positivos,prop_test,tasa_fp,prop_train
sttl,,,,,
254,15049,8099,0.407,0.538,0.201
62,4129,1182,0.112,0.286,0.039
32,19,2,0.001,0.105,0.000
0,987,0,0.027,0.000,0.051
1,38,0,0.001,0.000,0.001
31,16702,0,0.451,0.000,0.705


## 3. Distribución de `sttl` por clase en train y test

In [4]:
for nombre, df in [("train", train), ("test", test)]:
    valores = df["sttl"].where(df["sttl"].isin([0, 31, 62, 254]), "otro")
    t = pd.crosstab(valores, df["label"], normalize="columns").round(3)
    t.columns = ["normal", "ataque"]
    print(nombre)
    print(t, "\n")

train
      normal  ataque
sttl                
0      0.051   0.003
31     0.705   0.000
62     0.039   0.130
254    0.201   0.867
otro   0.005   0.000 

test
      normal  ataque
sttl                
0      0.027   0.001
31     0.451   0.000
62     0.112   0.138
254    0.407   0.860
otro   0.004   0.001 


**Interpretación:** en el train, `sttl = 254` aparece en el 87 % de los ataques y solo en el 20 % del tráfico normal,
así que el modelo lo aprendió como señal de ataque. En el test, el tráfico normal con `sttl = 254` se duplica (41 %),
y de ahí sale el 87 % de los falsos positivos. Los normales con `sttl = 31` no generan ninguna falsa alarma.
Es el sesgo del TTL que marcamos como limitación del dataset en el Charter.

## 4. ¿Cómo clasifica esas falsas alarmas el modelo multiclase?

In [5]:
bundle_multi = joblib.load(MODELS_DIR / "multiclass_xgb.joblib")
X_multi, _ = get_X_y(test, "multiclass")
categoria = bundle_multi["label_encoder"].inverse_transform(bundle_multi["pipeline"].predict(X_multi))
es_falso_positivo = (y_test.values == 0) & (pred == 1)
pd.Series(categoria[es_falso_positivo]).value_counts().head(5)

Fuzzers      7402
Analysis     1115
Exploits      334
Shellcode     304
DoS            60
Name: count, dtype: int64

**Interpretación:** la mayoría de las falsas alarmas el modelo multiclase las clasifica como Fuzzers.
Por eso en el test baja el F1 de Normal y la precisión de Fuzzers.